# Roulette Simulator
A full-featured roulette simulator with European and American variants, interactive graphical board, chip placement, probability calculations, and session logging.

In [ ]:
import random
import math
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, Circle, Wedge
from matplotlib.collections import PatchCollection
from IPython.display import display, clear_output, HTML
import datetime
import json

In [ ]:
ROULETTE_CONFIG = {
    'european': {
        'numbers': list(range(0, 37)),
        'green': [0],
        'red': [1,3,5,7,9,12,14,16,18,19,21,23,25,27,30,32,34,36],
        'black': [2,4,6,8,10,11,13,15,17,20,22,24,26,28,29,31,33,35],
        'zero_slots': ['0'],
        'house_edge': 2.7,
    },
    'american': {
        'numbers': list(range(0, 37)) + [37],
        'green': [0, 37],
        'red': [1,3,5,7,9,12,14,16,18,19,21,23,25,27,30,32,34,36],
        'black': [2,4,6,8,10,11,13,15,17,20,22,24,26,28,29,31,33,35],
        'zero_slots': ['0', '00'],
        'house_edge': 5.26,
    }
}

EURO_WHEEL_ORDER = [0,32,15,19,4,21,2,25,17,34,6,27,13,36,11,30,8,23,10,5,24,16,33,1,20,14,31,9,22,18,29,7,28,12,35,3,26]
AMERICAN_WHEEL_ORDER = [0,28,9,26,30,11,7,20,32,17,5,22,34,15,3,24,36,13,1,37,27,10,25,29,12,8,19,31,18,6,21,33,16,4,23,35,14,2]

BET_TYPES = {
    'straight': {'description': 'Single number', 'payout': 35, 'numbers_count': 1},
    'split': {'description': 'Two adjacent numbers', 'payout': 17, 'numbers_count': 2},
    'street': {'description': 'Row of three', 'payout': 11, 'numbers_count': 3},
    'corner': {'description': 'Four number square', 'payout': 8, 'numbers_count': 4},
    'six_line': {'description': 'Two rows of three', 'payout': 5, 'numbers_count': 6},
    'column': {'description': 'Column of 12', 'payout': 2, 'numbers_count': 12},
    'dozen': {'description': 'Group of 12', 'payout': 2, 'numbers_count': 12},
    'red': {'description': 'All red numbers', 'payout': 1, 'numbers_count': 18},
    'black': {'description': 'All black numbers', 'payout': 1, 'numbers_count': 18},
    'odd': {'description': 'All odd numbers', 'payout': 1, 'numbers_count': 18},
    'even': {'description': 'All even numbers', 'payout': 1, 'numbers_count': 18},
    'low': {'description': '1-18', 'payout': 1, 'numbers_count': 18},
    'high': {'description': '19-36', 'payout': 1, 'numbers_count': 18},
}

def number_label(n):
    if n == 37:
        return '00'
    return str(n)

In [ ]:
class RouletteGame:
    def __init__(self, variant='european'):
        self.variant = variant
        self.config = ROULETTE_CONFIG[variant]
        self.numbers = self.config['numbers']
        self.total_slots = len(self.numbers)
        self.bets = []
        self.session_log = []
        self.balance = 1000
        self.spin_count = 0

    def set_balance(self, amount):
        self.balance = amount

    def place_bet(self, bet_type, numbers, amount):
        if amount > self.balance:
            return False, 'Insufficient balance'
        if bet_type not in BET_TYPES:
            return False, f'Unknown bet type: {bet_type}'
        winning_numbers = []
        if bet_type == 'straight':
            winning_numbers = numbers[:1]
        elif bet_type == 'split':
            winning_numbers = numbers[:2]
        elif bet_type == 'street':
            winning_numbers = numbers[:3]
        elif bet_type == 'corner':
            winning_numbers = numbers[:4]
        elif bet_type == 'six_line':
            winning_numbers = numbers[:6]
        elif bet_type == 'column':
            col = numbers[0]
            winning_numbers = [n for n in range(1, 37) if n % 3 == (col % 3) or (col % 3 == 0 and n % 3 == 0)]
            if col % 3 == 1:
                winning_numbers = [n for n in range(1, 37) if n % 3 == 1]
            elif col % 3 == 2:
                winning_numbers = [n for n in range(1, 37) if n % 3 == 2]
            else:
                winning_numbers = [n for n in range(1, 37) if n % 3 == 0]
        elif bet_type == 'dozen':
            d = numbers[0]
            if d == 1:
                winning_numbers = list(range(1, 13))
            elif d == 2:
                winning_numbers = list(range(13, 25))
            else:
                winning_numbers = list(range(25, 37))
        elif bet_type == 'red':
            winning_numbers = self.config['red']
        elif bet_type == 'black':
            winning_numbers = self.config['black']
        elif bet_type == 'odd':
            winning_numbers = [n for n in range(1, 37) if n % 2 == 1]
        elif bet_type == 'even':
            winning_numbers = [n for n in range(1, 37) if n % 2 == 0]
        elif bet_type == 'low':
            winning_numbers = list(range(1, 19))
        elif bet_type == 'high':
            winning_numbers = list(range(19, 37))
        else:
            winning_numbers = numbers

        probability = len(winning_numbers) / self.total_slots
        payout_ratio = BET_TYPES[bet_type]['payout']

        bet = {
            'type': bet_type,
            'numbers': winning_numbers,
            'amount': amount,
            'payout_ratio': payout_ratio,
            'probability': probability,
            'labels': [number_label(n) for n in winning_numbers],
        }
        self.bets.append(bet)
        return True, bet

    def clear_bets(self):
        self.bets = []

    def spin(self):
        result = random.choice(self.numbers)
        self.spin_count += 1
        total_wagered = sum(b['amount'] for b in self.bets)
        self.balance -= total_wagered
        total_won = 0
        bet_results = []
        for bet in self.bets:
            won = result in bet['numbers']
            winnings = 0
            if won:
                winnings = bet['amount'] * bet['payout_ratio'] + bet['amount']
                total_won += winnings
            bet_results.append({
                'type': bet['type'],
                'labels': bet['labels'],
                'amount': bet['amount'],
                'won': won,
                'winnings': winnings,
                'probability': bet['probability'],
            })
        self.balance += total_won
        net = total_won - total_wagered

        color = 'green'
        if result in self.config['red']:
            color = 'red'
        elif result in self.config['black']:
            color = 'black'

        spin_record = {
            'spin': self.spin_count,
            'time': datetime.datetime.now().strftime('%H:%M:%S'),
            'result': number_label(result),
            'result_num': result,
            'color': color,
            'total_wagered': total_wagered,
            'total_won': total_won,
            'net': net,
            'balance': self.balance,
            'bets': bet_results,
        }
        self.session_log.append(spin_record)
        self.clear_bets()
        return spin_record

    def get_probability_summary(self):
        if not self.bets:
            return 'No bets placed'
        lines = []
        for b in self.bets:
            lines.append(f"  {b['type'].upper()} on {', '.join(b['labels'][:5])}{'...' if len(b['labels'])>5 else ''}: "
                         f"${b['amount']} | Win prob: {b['probability']*100:.1f}% | Payout: {b['payout_ratio']}:1")
        return '\n'.join(lines)

print('RouletteGame class loaded.')

In [ ]:
def display_session_log(game):
    if not game.session_log:
        print('No spins yet.')
        return
    color_map = {'red': '#E53935', 'black': '#263238', 'green': '#43A047'}
    total_wagered = sum(r['total_wagered'] for r in game.session_log)
    total_won_all = sum(r['total_won'] for r in game.session_log)
    total_net = sum(r['net'] for r in game.session_log)
    wins = sum(1 for r in game.session_log if r['net'] > 0)
    losses = sum(1 for r in game.session_log if r['net'] < 0)
    pushes = sum(1 for r in game.session_log if r['net'] == 0)
    net_color_summary = '#43A047' if total_net >= 0 else '#E53935'

    html = '''
    <style>
    .roulette-log { font-family: 'Segoe UI', Arial, sans-serif; border-radius: 12px; overflow: hidden; border: 1px solid #37474F; margin: 10px 0; }
    .roulette-log-header { background: linear-gradient(135deg, #1a237e 0%, #283593 100%); padding: 16px 20px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px; }
    .roulette-log-header h3 { margin: 0; color: #FFD54F; font-size: 18px; letter-spacing: 1px; }
    .stats-pills { display: flex; gap: 8px; flex-wrap: wrap; }
    .stat-pill { padding: 4px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: white; }
    .pill-win { background: #2E7D32; }
    .pill-loss { background: #C62828; }
    .pill-push { background: #546E7A; }
    .pill-net { background: ''' + net_color_summary + '''; }
    .roulette-log table { width: 100%; border-collapse: collapse; font-size: 13px; }
    .roulette-log thead th { background: #1a237e; color: #B3E5FC; padding: 10px 8px; text-align: center; font-size: 11px; text-transform: uppercase; letter-spacing: 0.5px; border-bottom: 2px solid #FFD54F; }
    .roulette-log tbody tr { transition: background 0.15s; }
    .roulette-log tbody tr:nth-child(even) { background: #E8EAF6; }
    .roulette-log tbody tr:nth-child(odd) { background: #F5F5F5; }
    .roulette-log tbody tr:hover { background: #C5CAE9; }
    .roulette-log td { padding: 8px 6px; text-align: center; vertical-align: middle; border-bottom: 1px solid #CFD8DC; }
    .result-badge { display: inline-block; min-width: 32px; padding: 3px 10px; border-radius: 16px; color: white; font-weight: 700; font-size: 14px; letter-spacing: 0.5px; text-shadow: 0 1px 2px rgba(0,0,0,0.3); }
    .net-positive { color: #2E7D32; font-weight: 700; }
    .net-negative { color: #C62828; font-weight: 700; }
    .net-zero { color: #546E7A; font-weight: 600; }
    .bet-chip { display: inline-block; padding: 2px 8px; border-radius: 10px; font-size: 11px; margin: 1px 2px; font-weight: 600; }
    .bet-win { background: #E8F5E9; color: #2E7D32; border: 1px solid #A5D6A7; }
    .bet-loss { background: #FFEBEE; color: #C62828; border: 1px solid #EF9A9A; }
    .balance-cell { font-weight: 700; font-size: 14px; color: #1a237e; }
    .roulette-log-footer { background: #E8EAF6; padding: 10px 20px; font-size: 12px; color: #455A64; text-align: right; border-top: 2px solid #C5CAE9; }
    </style>
    '''

    html += '<div class="roulette-log">'
    html += '<div class="roulette-log-header">'
    html += '<h3>Session Log</h3>'
    html += '<div class="stats-pills">'
    html += f'<span class="stat-pill pill-win">W {wins}</span>'
    html += f'<span class="stat-pill pill-loss">L {losses}</span>'
    if pushes:
        html += f'<span class="stat-pill pill-push">P {pushes}</span>'
    net_sign = "+" if total_net >= 0 else ""
    html += f'<span class="stat-pill pill-net">Net {net_sign}${total_net}</span>'
    html += '</div></div>'

    html += '<div style="max-height:400px; overflow-y:auto;">'
    html += '<table><thead><tr>'
    html += '<th>#</th><th>Time</th><th>Result</th><th>Wagered</th><th>Won</th><th>Net</th><th>Balance</th><th>Bets</th>'
    html += '</tr></thead><tbody>'

    for rec in game.session_log:
        c = color_map.get(rec['color'], '#546E7A')
        if rec['net'] > 0:
            net_class = 'net-positive'
            net_str = f'+${rec["net"]}'
        elif rec['net'] < 0:
            net_class = 'net-negative'
            net_str = f'-${abs(rec["net"])}'
        else:
            net_class = 'net-zero'
            net_str = '$0'

        bet_chips = []
        for b in rec['bets']:
            chip_class = 'bet-win' if b['won'] else 'bet-loss'
            icon = '\u2713' if b['won'] else '\u2717'
            bet_chips.append(f'<span class="bet-chip {chip_class}">{icon} {b["type"]} ${b["amount"]}</span>')

        html += '<tr>'
        html += f'<td style="font-weight:600; color:#455A64;">{rec["spin"]}</td>'
        html += f'<td style="color:#78909C; font-size:12px;">{rec["time"]}</td>'
        html += f'<td><span class="result-badge" style="background:{c};">{rec["result"]}</span></td>'
        html += f'<td>${rec["total_wagered"]}</td>'
        html += f'<td>${rec["total_won"]}</td>'
        html += f'<td class="{net_class}">{net_str}</td>'
        html += f'<td class="balance-cell">${rec["balance"]}</td>'
        html += f'<td style="text-align:left;"> {" ".join(bet_chips)}</td>'
        html += '</tr>'

    html += '</tbody></table></div>'
    html += f'<div class="roulette-log-footer">Total wagered: ${total_wagered} &middot; Total returned: ${total_won_all} &middot; {len(game.session_log)} spins</div>'
    html += '</div>'
    display(HTML(html))

def display_balance_chart(game):
    if len(game.session_log) < 2:
        return
    spins = [0] + [r['spin'] for r in game.session_log]
    balances = [game.session_log[0]['balance'] - game.session_log[0]['net']] + [r['balance'] for r in game.session_log]
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(spins, balances, color='#1565C0', linewidth=2, marker='o', markersize=4)
    ax.fill_between(spins, balances, alpha=0.1, color='#1565C0')
    ax.axhline(y=balances[0], color='gray', linestyle='--', alpha=0.5, label='Starting')
    ax.set_xlabel('Spin #')
    ax.set_ylabel('Balance ($)')
    ax.set_title('Balance Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print('Session log display functions loaded.')

In [ ]:
class RouletteBoard:
    def __init__(self, game):
        self.game = game
        self.selected_numbers = set()
        self.chip_amounts = {}
        self.fig = None
        self.ax = None

    def get_color(self, n):
        if n in self.game.config['green']:
            return '#2E7D32'
        elif n in self.game.config['red']:
            return '#C62828'
        else:
            return '#212121'

    def draw_board(self, result=None, highlight_bets=None):
        fig, ax = plt.subplots(figsize=(14, 8))
        self.fig = fig
        self.ax = ax
        ax.set_xlim(-1, 16)
        ax.set_ylim(-2.5, 5.5)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_facecolor('#1B5E20')
        fig.patch.set_facecolor('#1B5E20')

        bg = FancyBboxPatch((-0.7, -2.2), 16.2, 7.2, boxstyle='round,pad=0.2',
                            facecolor='#1B5E20', edgecolor='#FFD700', linewidth=2)
        ax.add_patch(bg)

        zero_x = 0
        if self.game.variant == 'american':
            rect_0 = FancyBboxPatch((zero_x - 0.45, 1.55), 0.9, 1.35,
                                     boxstyle='round,pad=0.05', facecolor='#2E7D32',
                                     edgecolor='#FFD700', linewidth=1.5)
            ax.add_patch(rect_0)
            ax.text(zero_x, 2.25, '0', ha='center', va='center',
                    fontsize=14, fontweight='bold', color='white')
            if 0 in self.selected_numbers:
                ax.plot(zero_x, 2.25, 'o', color='#FFD700', markersize=18, zorder=5)
                ax.text(zero_x, 2.25, str(self.chip_amounts.get(0, '')),
                        ha='center', va='center', fontsize=8, color='black', fontweight='bold', zorder=6)

            rect_00 = FancyBboxPatch((zero_x - 0.45, 0.05), 0.9, 1.35,
                                      boxstyle='round,pad=0.05', facecolor='#2E7D32',
                                      edgecolor='#FFD700', linewidth=1.5)
            ax.add_patch(rect_00)
            ax.text(zero_x, 0.75, '00', ha='center', va='center',
                    fontsize=14, fontweight='bold', color='white')
            if 37 in self.selected_numbers:
                ax.plot(zero_x, 0.75, 'o', color='#FFD700', markersize=18, zorder=5)
                ax.text(zero_x, 0.75, str(self.chip_amounts.get(37, '')),
                        ha='center', va='center', fontsize=8, color='black', fontweight='bold', zorder=6)
        else:
            rect_0 = FancyBboxPatch((zero_x - 0.45, 0.05), 0.9, 2.9,
                                     boxstyle='round,pad=0.05', facecolor='#2E7D32',
                                     edgecolor='#FFD700', linewidth=1.5)
            ax.add_patch(rect_0)
            ax.text(zero_x, 1.5, '0', ha='center', va='center',
                    fontsize=16, fontweight='bold', color='white')
            if 0 in self.selected_numbers:
                ax.plot(zero_x, 1.5, 'o', color='#FFD700', markersize=18, zorder=5)
                ax.text(zero_x, 1.5, str(self.chip_amounts.get(0, '')),
                        ha='center', va='center', fontsize=8, color='black', fontweight='bold', zorder=6)

        for i in range(1, 37):
            col = (i - 1) // 3
            row = (i - 1) % 3
            x = col + 1
            y = 2 - row
            color = self.get_color(i)
            is_result = (result is not None and result == i)
            edge_color = '#FFD700' if is_result else '#8D6E63'
            lw = 3 if is_result else 1

            rect = FancyBboxPatch((x - 0.45, y - 0.45), 0.9, 0.9,
                                   boxstyle='round,pad=0.05', facecolor=color,
                                   edgecolor=edge_color, linewidth=lw)
            ax.add_patch(rect)
            ax.text(x, y, str(i), ha='center', va='center',
                    fontsize=11, fontweight='bold', color='white')

            if i in self.selected_numbers:
                ax.plot(x, y + 0.2, 'o', color='#FFD700', markersize=16, zorder=5)
                ax.text(x, y + 0.2, str(self.chip_amounts.get(i, '')),
                        ha='center', va='center', fontsize=7, color='black', fontweight='bold', zorder=6)

            if is_result:
                ax.plot(x, y, '*', color='#FFD700', markersize=20, zorder=5)

        outside_y = -1.0
        outside_bets = [
            (2.5, '1st 12', '#5D4037'), (6.5, '2nd 12', '#5D4037'), (10.5, '3rd 12', '#5D4037'),
        ]
        for ox, label, color in outside_bets:
            rect = FancyBboxPatch((ox - 1.45, outside_y - 0.35), 3.9, 0.7,
                                   boxstyle='round,pad=0.05', facecolor=color,
                                   edgecolor='#8D6E63', linewidth=1)
            ax.add_patch(rect)
            ax.text(ox + 0.5, outside_y, label, ha='center', va='center',
                    fontsize=10, fontweight='bold', color='white')

        even_money_y = -1.8
        even_money = [
            (1.5, '1-18', '#5D4037'), (3.5, 'EVEN', '#5D4037'),
            (5.5, 'RED', '#C62828'), (7.5, 'BLACK', '#212121'),
            (9.5, 'ODD', '#5D4037'), (11.5, '19-36', '#5D4037'),
        ]
        for ox, label, color in even_money:
            rect = FancyBboxPatch((ox - 0.95, even_money_y - 0.3), 1.9, 0.6,
                                   boxstyle='round,pad=0.05', facecolor=color,
                                   edgecolor='#8D6E63', linewidth=1)
            ax.add_patch(rect)
            ax.text(ox, even_money_y, label, ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')

        col_labels_x = 13.2
        for row, label in [(2, '2:1'), (1, '2:1'), (0, '2:1')]:
            y = 2 - row
            rect = FancyBboxPatch((col_labels_x - 0.4, y - 0.35), 0.8, 0.7,
                                   boxstyle='round,pad=0.05', facecolor='#5D4037',
                                   edgecolor='#8D6E63', linewidth=1)
            ax.add_patch(rect)
            ax.text(col_labels_x, y, label, ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')

        title = f"{'American' if self.game.variant == 'american' else 'European'} Roulette"
        ax.text(6.5, 4.5, title, ha='center', va='center',
                fontsize=18, fontweight='bold', color='#FFD700',
                fontfamily='serif')

        ax.text(6.5, 3.8, f'Balance: ${self.game.balance}', ha='center', va='center',
                fontsize=13, color='white', fontweight='bold')

        if result is not None:
            result_color = self.get_color(result)
            circle = Circle((14.5, 4.0), 0.6, facecolor=result_color, edgecolor='#FFD700', linewidth=2)
            ax.add_patch(circle)
            ax.text(14.5, 4.0, number_label(result), ha='center', va='center',
                    fontsize=16, fontweight='bold', color='white')
            ax.text(14.5, 3.0, 'RESULT', ha='center', va='center',
                    fontsize=9, color='#FFD700', fontweight='bold')

        plt.tight_layout()
        plt.show()
        self.fig = fig
        return fig

    def select_number(self, number, chip_amount=10):
        self.selected_numbers.add(number)
        self.chip_amounts[number] = self.chip_amounts.get(number, 0) + chip_amount

    def deselect_number(self, number):
        self.selected_numbers.discard(number)
        self.chip_amounts.pop(number, None)

    def clear_selections(self):
        self.selected_numbers.clear()
        self.chip_amounts.clear()

    def place_selected_bets(self):
        results = []
        for n in self.selected_numbers:
            amount = self.chip_amounts.get(n, 10)
            ok, bet = self.game.place_bet('straight', [n], amount)
            results.append((n, ok, bet))
        return results

print('RouletteBoard class loaded.')

In [ ]:
def play_round(game, board, bets_config):
    board.clear_selections()
    print('='*60)
    print(f"Spin #{game.spin_count + 1} | Balance: ${game.balance}")
    print('='*60)

    for bet in bets_config:
        if bet.get('numbers_select'):
            for n in bet['numbers_select']:
                board.select_number(n, bet.get('chip', 10))

        if bet.get('bet_type'):
            nums = bet.get('numbers', [])
            ok, result = game.place_bet(bet['bet_type'], nums, bet.get('chip', 10))
            if ok:
                label = bet['bet_type'].upper()
                if nums:
                    label += f" on {[number_label(n) for n in nums[:5]]}"
                print(f"  Placed: {label} - ${bet.get('chip', 10)}")
            else:
                print(f"  FAILED: {result}")
        elif bet.get('numbers_select'):
            for n in bet['numbers_select']:
                ok, result = game.place_bet('straight', [n], bet.get('chip', 10))
                if ok:
                    print(f"  Placed: STRAIGHT on {number_label(n)} - ${bet.get('chip', 10)}")
                else:
                    print(f"  FAILED: {result}")

    print()
    print('Bet Probabilities:')
    print(game.get_probability_summary())
    print()

    board.draw_board()

    spin_result = game.spin()

    print(f"\n  >>> RESULT: {spin_result['result']} ({spin_result['color'].upper()}) <<<")
    print()
    for b in spin_result['bets']:
        status = 'WIN' if b['won'] else 'LOSS'
        print(f"  [{status}] {b['type'].upper()}: ${b['amount']} -> ${b['winnings']}")
    print(f"\n  Net: {'+' if spin_result['net'] >= 0 else ''}{spin_result['net']}")
    print(f"  Balance: ${spin_result['balance']}")
    print()

    board.clear_selections()
    board.draw_board(result=spin_result['result_num'])

    return spin_result

print('play_round function loaded.')

## Interactive Roulette - Choose Your Variant
Run the cell below to start a game. Modify `variant` to `'american'` or `'european'`.

Then use the **Place Bets & Spin** cell to play rounds. Edit the `bets` list to place chips on your chosen numbers.

In [ ]:
variant = 'european'
starting_balance = 1000

game = RouletteGame(variant=variant)
game.set_balance(starting_balance)
board = RouletteBoard(game)

print(f'Game created: {variant.upper()} Roulette')
print(f'Starting balance: ${starting_balance}')
print(f'House edge: {game.config["house_edge"]}%')
print(f'Total slots: {game.total_slots}')
print()
board.draw_board()

## Place Bets & Spin
Edit the `bets` list below to place your chips. You can:
- **Pick individual numbers**: `{'numbers_select': [7, 14, 21], 'chip': 25}` — places straight bets on each
- **Named bets**: `{'bet_type': 'red', 'numbers': [], 'chip': 50}` — outside bet on red
- **Corners/splits**: `{'bet_type': 'corner', 'numbers': [1,2,4,5], 'chip': 20}`

### Bet Types
| Type | Description | Payout |
|------|-------------|--------|
| straight | Single number | 35:1 |
| split | Two adjacent | 17:1 |
| street | Row of 3 | 11:1 |
| corner | Block of 4 | 8:1 |
| six_line | Two rows | 5:1 |
| column | Column (pass col 1/2/3) | 2:1 |
| dozen | 1st/2nd/3rd 12 (pass 1/2/3) | 2:1 |
| red/black | Color | 1:1 |
| odd/even | Parity | 1:1 |
| low/high | 1-18 / 19-36 | 1:1 |

In [ ]:
bets = [
    {'numbers_select': [7, 17, 32], 'chip': 25},
    {'bet_type': 'red', 'numbers': [], 'chip': 50},
    {'bet_type': 'corner', 'numbers': [1, 2, 4, 5], 'chip': 20},
]

result = play_round(game, board, bets)

## Session Log & Analytics
Run the cell below after a few spins to see your session history and balance chart.

In [ ]:
display_session_log(game)
display_balance_chart(game)

## Quick Multi-Spin
Run multiple rounds with the same bets to simulate a session.

In [ ]:
num_spins = 10
bet_template = [
    {'numbers_select': [0, 17, 26], 'chip': 10},
    {'bet_type': 'black', 'numbers': [], 'chip': 25},
]

for i in range(num_spins):
    for bet in bet_template:
        if bet.get('numbers_select'):
            for n in bet['numbers_select']:
                game.place_bet('straight', [n], bet.get('chip', 10))
        elif bet.get('bet_type'):
            game.place_bet(bet['bet_type'], bet.get('numbers', []), bet.get('chip', 10))
    spin_result = game.spin()
    net_str = f"+{spin_result['net']}" if spin_result['net'] >= 0 else str(spin_result['net'])
    print(f"Spin {spin_result['spin']:>3}: {spin_result['result']:>3} ({spin_result['color']:>5}) | "
          f"Wagered: ${spin_result['total_wagered']:<6} Won: ${spin_result['total_won']:<6} "
          f"Net: {net_str:>6} | Balance: ${spin_result['balance']}")

print(f"\nFinal balance after {num_spins} quick spins: ${game.balance}")
print()
display_session_log(game)
display_balance_chart(game)